In [1]:
# python libraries imports
import pandas as pd
import numpy as np

# customed local functions imports
import sys
sys.path.append("./scripts")

from utils import *

#### **Dataset load and details**

In [3]:
ratings = load_amazonMusic()
print("============ Amazon Music Dataset ===========")
print(f"#Ratings: {ratings["rating"].count()} | #Users: {ratings['user'].nunique()} | #Items: {ratings['item'].nunique()} ")

============ Amazon Music Dataset ===========
#Ratings: 128763 | #Users: 100952 | #Items: 70511 


**Effectiveness**

In [4]:
results = []

distances = ["l1", "l2"]
aggregates = ["avg", "max", "min"]

# Compute once, it doesn't depend on the variant
aa = average_ranking(ratings)

for distance in distances:
    for aggregate in aggregates:
        rep = reputation_ranking(ratings, distance, aggregate, lam=0.2)
        tau = kendall(rep, aa)

        results.append({
            "algorithm": f"{distance.upper()}-{aggregate.upper()}",
            "kendall_tau": tau,
        })

df_clean = pd.DataFrame(results)
display(df_clean)

,algorithm,kendall_tau
0,L1-AVG,0.994885
1,L1-MAX,0.834069
2,L1-MIN,0.994983
3,L2-AVG,0.993701
4,L2-MAX,0.840467
5,L2-MIN,0.995954


**Robustness**

In [5]:
results = []

distances = ["l1", "l2"]
aggregates = ["avg", "max", "min"]
spam_ratios = [0.1, 0.3, 0.5]

# Noisy datasets: generated once per ratio, shared by all algorithms
noisy_datasets = {p: add_spammers(ratings, p) for p in spam_ratios}

# Baseline (AA): original vs noisy
aa_orig = average_ranking(ratings)
for p, noisy in noisy_datasets.items():
    results.append({
        "algorithm": "AA",
        "spam_ratio": p,
        "kendall_tau": kendall(aa_orig, average_ranking(noisy)),
    })

# Reputation-based algorithms: original vs noisy
for distance in distances:
    for aggregate in aggregates:
        rep_orig = reputation_ranking(ratings, distance, aggregate, lam=0.2)

        for p, noisy in noisy_datasets.items():
            rep_noisy = reputation_ranking(noisy, distance, aggregate, lam=0.2)

            results.append({
                "algorithm": f"{distance.upper()}-{aggregate.upper()}",
                "spam_ratio": p,
                "kendall_tau": kendall(rep_orig, rep_noisy),
            })

df_robust = pd.DataFrame(results)

# Table: one row per algorithm, one column per spam ratio
display(df_robust.pivot(index="algorithm", columns="spam_ratio", values="kendall_tau"))

spam_ratio,0.1,0.3,0.5
algorithm,,,
AA,0.842275,0.642203,0.514555
L1-AVG,0.842788,0.643593,0.516673
L1-MAX,0.814890,0.609021,0.477651
L1-MIN,0.834660,0.628367,0.498938
L2-AVG,0.843710,0.645084,0.517984
L2-MAX,0.818610,0.612816,0.480544
L2-MIN,0.836642,0.632687,0.504586
